In [0]:
WITH

ref_file_data AS (
  SELECT
    hcp_npi,
    hcp_zip                     AS ref_file_zip,
    territory_id               AS ref_file_territory_id,
    territory                  AS ref_file_territory_name,
    region_id                  AS ref_file_region_id,
    region                     AS ref_file_region_name
  FROM cmpa_insights_internal_schema.reference_file
),

ref_file_zip_mapping AS (
  SELECT
    r.hcp_npi,
    z.territory_id             AS ref_zip_territory_id,
    z.territory_name           AS ref_zip_territory_name,
    z.region_id                AS ref_zip_region_id,
    z.region_name              AS ref_zip_region_name
  FROM cmpa_insights_internal_schema.reference_file r
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON TRY_CAST(r.hcp_zip AS STRING) = TRY_CAST(z.zipcode AS STRING)
  WHERE r.territory IS NULL
     OR TRIM(r.territory) IN ('', '-')
),

komodo_zip_mapping AS (
  SELECT
    p.NPI                      AS komodo_hcp_npi,
    p.PROVIDER_ZIP             AS komodo_zip,
    z.territory_id             AS komodo_territory_id,
    z.territory_name           AS komodo_territory_name
  FROM com_edp_prd.com_raw.kom_providers p
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON TRY_CAST(p.PROVIDER_ZIP AS STRING) = TRY_CAST(z.zipcode AS STRING)
  WHERE p.PROVIDER_TYPE = 'INDIVIDUAL'
),

hcp_input_list AS (
  SELECT explode(array(
    '1992801294', '1992790778', '1982674123', '1982649133', '1972868073', '1972765220', '1972762540', '1972712743', '1972590354', '1972554897', '1952575763', '1952465635', '1942543962', '1942247317', '1932429917', '1922098664', '1912902636', '1912296096', '1912252388', '1912087354', '1902935729', '1902830532', '1902012180', '1891891362', '1891767307', '1891413464', '1891174785', '1891054623', '1871734509', '1861897241', '1861888448', '1861866717', '1861629172', '1861623985', '1861582348', '1861445512', '1861430175', '1851403539', '1851331722', '1841551090', '1841486974', '1831109792', '1821479957', '1821309410', '1821225863', '1821019555', '1811955479', '1811934490', '1811277130', '1811160609', '1811128655', '1811075567', '1801892401', '1801842133', '1801812128', '1801811351', '1801088711', '1790715134', '1780926006', '1780833483', '1780817700', '1780635839', '1770579872', '1760689061', '1760652267', '1760561336', '1760473524', '1750489811', '1750317053', '1740400506', '1740218296', '1740208750', '1740202159', '1730355579', '1730239153', '1720407166', '1720378904', '1720127194', '1710937982', '1710063474', '1700872710', '1700847860', '1700044336', '1699983155', '1699824730', '1699798603', '1699743088', '1689734741', '1689699381', '1689651218', '1679893010', '1679869143', '1679839914', '1679546139', '1669821781', '1669638961', '1669450896', '1659717148', '1659459444', '1649478553', '1629183686', '1629050851', '1619979697', '1609864891', '1609003011', '1598708794', '1588984561', '1588827026', '1588735005', '1588698088', '1588196463', '1578988994', '1578837134', '1578528816', '1578505483', '1568859411', '1568858264', '1568624633', '1568548246', '1568452233', '1568411346', '1558667915', '1558591867', '1558524629', '1558523845', '1548266257', '1548216690', '1538253570', '1528174885', '1518629294', '1518308204', '1518082304', '1508125295', '1497118004', '1497071963', '1497045298', '1497016182', '1487958146', '1477999522', '1477901221', '1467669721', '1467415034', '1457745341', '1457439713', '1457330607', '1457318545', '1447366794', '1447348388', '1447281878', '1447270087', '1447202163', '1437152832', '1427438811', '1417593237', '1417132804', '1407878796', '1407876584', '1407868896', '1407172521', '1396239992', '1396006946', '1386686467', '1376764779', '1376705699', '1376663187', '1376098095', '1366935439', '1366829152', '1366671380', '1366407488', '1356606206', '1356429518', '1356394324', '1356335327', '1336301977', '1336260280', '1336259670', '1336248996', '1326420662', '1326085010', '1316916711', '1316464001', '1306990817', '1306926944', '1306838735', '1306011606', '1295815751', '1295709913', '1295113835', '1285710269', '1285600221', '1285098426', '1275612749', '1275051534', '1265782817', '1265588925', '1255640058', '1255623815', '1255435301', '1255427324', '1255422531', '1245211978', '1225476930', '1225396930', '1225133812', '1225113954', '1215983382', '1205896933', '1205814506', '1205068392', '1194986554', '1194915306', '1194833376', '1194776641', '1194067470', '1194043307', '1184929101', '1174658272', '1164957437', '1164765616', '1164733960', '1164628053', '1164470126', '1154634566', '1154521722', '1154515963', '1154431567', '1154428027', '1144481359', '1144230764', '1134549074', '1134534597', '1134488471', '1134395817', '1134199946', '1134149495', '1134112436', '1124438809', '1124405311', '1124398847', '1124348107', '1114992856', '1114975737', '1114949617', '1114360997', '1114294477', '1114212370', '1114038551', '1104906445', '1104395656', '1104172584', '1104010982', '1104010776', '1083857858', '1073239018', '1063647071', '1053637850', '1053629881', '1053396135', '1043375363', '1033539671', '1033538996', '1033220314', '1023492980', '1023121159', '1013914423', '1013228410', '1013086735', '1003985649', '1003927500', '1003889049', '1003278540', '1003203779', '1003026139'
  )) AS hcp_npi
),

final_output AS (
  SELECT
    h.hcp_npi,

    -- Which source was used
    CASE
      WHEN r.ref_file_territory_name IS NOT NULL
       AND TRIM(r.ref_file_territory_name) NOT IN ('', '-')
        THEN 'Priority 1 - Reference File'
      WHEN r.hcp_npi IS NOT NULL
        THEN 'Priority 2 - Ref File ZIP Mapping'
      ELSE
        'Priority 3 - Komodo ZIP Mapping'
    END AS final_territory_source,

    -- Final Territory ID
    COALESCE(
      CASE
        WHEN r.ref_file_territory_name IS NOT NULL
         AND TRIM(r.ref_file_territory_name) NOT IN ('', '-')
        THEN CAST(r.ref_file_territory_id AS STRING)
      END,
      CAST(rz.ref_zip_territory_id AS STRING),
      CAST(k.komodo_territory_id AS STRING),
      'Unknown'
    ) AS final_territory_id,

    -- Final Territory Name
    COALESCE(
      CASE
        WHEN r.ref_file_territory_name IS NOT NULL
         AND TRIM(r.ref_file_territory_name) NOT IN ('', '-')
        THEN r.ref_file_territory_name
      END,
      rz.ref_zip_territory_name,
      k.komodo_territory_name,
      'Unknown'
    ) AS final_territory_name,

    -- Debug columns (very useful)
    r.ref_file_zip,
    r.ref_file_territory_name        AS raw_ref_file_territory,
    rz.ref_zip_territory_name        AS derived_ref_zip_territory,
    k.komodo_zip,
    k.komodo_territory_name          AS derived_komodo_territory

  FROM hcp_input_list h
  LEFT JOIN ref_file_data r
    ON TRY_CAST(h.hcp_npi AS STRING) = TRY_CAST(r.hcp_npi AS STRING)
  LEFT JOIN ref_file_zip_mapping rz
    ON TRY_CAST(h.hcp_npi AS STRING) = TRY_CAST(rz.hcp_npi AS STRING)
  LEFT JOIN komodo_zip_mapping k
    ON TRY_CAST(h.hcp_npi AS STRING) = TRY_CAST(k.komodo_hcp_npi AS STRING)
)

SELECT *
FROM final_output
ORDER BY final_territory_source, hcp_npi;